# F1 Action Gating Ablation Report

Date: 2026-04-26

This notebook reports the first complete three-way Action Gating ablation for `F1_code_qa`:

- `baseline`: terminal + file tools
- `file_only`: file read/search tools only
- `terminal_only`: terminal/process tools only

All metrics are read from `benchmark/results/f1_*.jsonl`. The code cells use only the Python standard library.

## Executive Summary

All three configs pass all F1 code-QA tasks, so the first-order quality signal is tied at 100%.

| Config | Runs | Passes | Pass Rate | Latency p50 | Latency p95 | Latency Max | Mean Input Tokens | Mean Output Tokens | Mean API Calls |
|---|---:|---:|---:|---:|---:|---:|---:|---:|---:|
| `baseline` | 30 | 30 | 1.000 | 12.483s | 42.178s | 46.366s | 3886.333 | 318.333 | 3.167 |
| `file_only` | 30 | 30 | 1.000 | 17.415s | 62.332s | 89.347s | 2821.033 | 260.267 | 3.333 |
| `terminal_only` | 30 | 30 | 1.000 | 16.035s | 49.601s | 60.561s | 2324.433 | 215.733 | 2.733 |

Current best early signal: `terminal_only` is the cheapest in tokens and API calls after adding repository-scope prompt guidance and applying terminal timeout from config. `baseline` remains fastest at p50, while `file_only` has the worst latency tail in this run.

## Experiment Design

| Axis | Held Constant | Varied |
|---|---|---|
| Task family | `F1_code_qa`, 10 tasks x 3 seeds | None |
| Model/provider | `kimi-k2.6` / `kimi-coding` | None |
| Observation shaping | Mostly off; benchmark repo-scope prompt now injected by runner | None for this ablation |
| Action gating | Same deny rules for clarify/delegation | Tool exposure: baseline vs file-only vs terminal-only |
| Transition control | `max_iterations=30`; terminal timeout sourced from config | None for this ablation |

Important correction during the run: terminal-only originally searched `/mnt` and `/mnt/c` because Hermes injects WSL environment hints. The runner now adds a benchmark-only repository-scope prompt and maps `timeout_per_tool_sec` to `TERMINAL_TIMEOUT`, so the final `terminal_only` results represent repo-local shell usage rather than host-filesystem scanning.

In [1]:
from __future__ import annotations

import json
from pathlib import Path

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent.parent

RESULTS_DIR = ROOT / "benchmark" / "results"
RESULT_FILES = sorted(RESULTS_DIR.glob("f1_*.jsonl"))

def percentile(values: list[float], pct: float) -> float | None:
    if not values:
        return None
    ordered = sorted(values)
    if len(ordered) == 1:
        return ordered[0]
    rank = (len(ordered) - 1) * pct
    lower = int(rank)
    upper = min(lower + 1, len(ordered) - 1)
    weight = rank - lower
    return ordered[lower] * (1 - weight) + ordered[upper] * weight

def mean(values: list[float]) -> float | None:
    return sum(values) / len(values) if values else None

def config_name(path: Path) -> str:
    return path.stem.removeprefix("f1_")

rows = []
for path in RESULT_FILES:
    config = config_name(path)
    for line in path.read_text().splitlines():
        if not line.strip():
            continue
        rec = json.loads(line)
        metrics = rec.get("metrics") or {}
        rows.append({
            "config": config,
            "task_id": rec.get("task_id"),
            "seed": rec.get("seed"),
            "q_pass": metrics.get("q_oracle_pass") is True,
            "latency_s": float(metrics.get("l_wall_clock_sec") or 0),
            "input_tokens": int(metrics.get("c_input_tokens") or 0),
            "output_tokens": int(metrics.get("c_output_tokens") or 0),
            "reasoning_tokens": int(metrics.get("c_reasoning_tokens") or 0),
            "api_calls": int(rec.get("api_calls") or 0),
            "stop_reason": rec.get("stop_reason"),
        })

len(rows), [p.name for p in RESULT_FILES]


(90, ['f1_baseline.jsonl', 'f1_file_only.jsonl', 'f1_terminal_only.jsonl'])

In [2]:
summary = []
for config in sorted({row["config"] for row in rows}):
    subset = [row for row in rows if row["config"] == config]
    latencies = [row["latency_s"] for row in subset]
    summary.append({
        "config": config,
        "runs": len(subset),
        "passes": sum(row["q_pass"] for row in subset),
        "pass_rate": mean([1.0 if row["q_pass"] else 0.0 for row in subset]),
        "latency_mean": mean(latencies),
        "latency_p50": percentile(latencies, 0.50),
        "latency_p95": percentile(latencies, 0.95),
        "latency_max": max(latencies),
        "input_tokens_mean": mean([row["input_tokens"] for row in subset]),
        "output_tokens_mean": mean([row["output_tokens"] for row in subset]),
        "api_calls_mean": mean([row["api_calls"] for row in subset]),
        "input_tokens_total": sum(row["input_tokens"] for row in subset),
        "output_tokens_total": sum(row["output_tokens"] for row in subset),
    })

for row in summary:
    print(
        f"{row['config']:15} runs={row['runs']:2} pass={row['passes']:2}/{row['runs']} "
        f"p50={row['latency_p50']:.3f}s p95={row['latency_p95']:.3f}s max={row['latency_max']:.3f}s "
        f"input_mean={row['input_tokens_mean']:.1f} output_mean={row['output_tokens_mean']:.1f} "
        f"api_mean={row['api_calls_mean']:.2f}"
    )


baseline        runs=30 pass=30/30 p50=12.483s p95=42.178s max=46.366s input_mean=3886.3 output_mean=318.3 api_mean=3.17
file_only       runs=30 pass=30/30 p50=17.415s p95=62.332s max=89.347s input_mean=2821.0 output_mean=260.3 api_mean=3.33
terminal_only   runs=30 pass=30/30 p50=16.035s p95=49.601s max=60.561s input_mean=2324.4 output_mean=215.7 api_mean=2.73


## Cost and Latency Tradeoff

Relative to `baseline`:

| Config | Input Token Delta | Output Token Delta | p50 Latency Delta | p95 Latency Delta | API Call Delta |
|---|---:|---:|---:|---:|---:|
| `file_only` | -27.41% | -18.24% | +4.932s | +20.154s | +0.167 |
| `terminal_only` | -40.19% | -32.23% | +3.552s | +7.423s | -0.433 |

Interpretation: within F1, all configs preserve quality. `terminal_only` gives the strongest token reduction and fewer API calls, but its p50 latency is still slower than baseline. `file_only` saves tokens but shows the largest latency tail in this run.

In [3]:
baseline = next(row for row in summary if row["config"] == "baseline")
for row in summary:
    if row["config"] == "baseline":
        continue
    print(row["config"])
    print("  input token delta:", f"{(row['input_tokens_mean'] / baseline['input_tokens_mean'] - 1) * 100:.2f}%")
    print("  output token delta:", f"{(row['output_tokens_mean'] / baseline['output_tokens_mean'] - 1) * 100:.2f}%")
    print("  p50 latency delta:", f"{row['latency_p50'] - baseline['latency_p50']:.3f}s")
    print("  p95 latency delta:", f"{row['latency_p95'] - baseline['latency_p95']:.3f}s")
    print("  api call delta:", f"{row['api_calls_mean'] - baseline['api_calls_mean']:.3f}")


file_only
  input token delta: -27.41%
  output token delta: -18.24%
  p50 latency delta: 4.932s
  p95 latency delta: 20.154s
  api call delta: 0.167
terminal_only
  input token delta: -40.19%
  output token delta: -32.23%
  p50 latency delta: 3.552s
  p95 latency delta: 7.423s
  api call delta: -0.433


## Slowest Runs

| Config | Task | Seed | Latency | Input Tokens | Output Tokens | API Calls |
|---|---|---:|---:|---:|---:|---:|
| `file_only` | `f1_07_approval_decisions` | 0 | 89.35s | 8395 | 546 | 3 |
| `file_only` | `f1_07_approval_decisions` | 2 | 68.05s | 13214 | 933 | 9 |
| `terminal_only` | `f1_02_iteration_budget_class` | 2 | 60.56s | 3977 | 379 | 4 |
| `terminal_only` | `f1_07_approval_decisions` | 0 | 57.62s | 15533 | 775 | 6 |
| `file_only` | `f1_01_version` | 2 | 55.34s | 676 | 235 | 4 |
| `file_only` | `f1_08_ctx_threshold` | 1 | 54.69s | 260 | 113 | 2 |
| `file_only` | `f1_07_approval_decisions` | 1 | 54.17s | 7650 | 734 | 9 |
| `baseline` | `f1_09_default_ctx_engine` | 2 | 46.37s | 3117 | 1090 | 7 |
| `baseline` | `f1_07_approval_decisions` | 1 | 43.70s | 25841 | 888 | 6 |
| `file_only` | `f1_04_loop_owned_count` | 1 | 42.67s | 1570 | 183 | 2 |

In [4]:
for row in sorted(rows, key=lambda item: item["latency_s"], reverse=True)[:10]:
    print(
        f"{row['config']:15} {row['task_id']:28} seed={row['seed']} "
        f"latency={row['latency_s']:.2f}s input={row['input_tokens']} "
        f"output={row['output_tokens']} api={row['api_calls']}"
    )


file_only       f1_07_approval_decisions     seed=0 latency=89.35s input=8395 output=546 api=3
file_only       f1_07_approval_decisions     seed=2 latency=68.05s input=13214 output=933 api=9
terminal_only   f1_02_iteration_budget_class seed=2 latency=60.56s input=3977 output=379 api=4
terminal_only   f1_07_approval_decisions     seed=0 latency=57.62s input=15533 output=775 api=6
file_only       f1_01_version                seed=2 latency=55.34s input=676 output=235 api=4
file_only       f1_08_ctx_threshold          seed=1 latency=54.69s input=260 output=113 api=2
file_only       f1_07_approval_decisions     seed=1 latency=54.17s input=7650 output=734 api=9
baseline        f1_09_default_ctx_engine     seed=2 latency=46.37s input=3117 output=1090 api=7
baseline        f1_07_approval_decisions     seed=1 latency=43.70s input=25841 output=888 api=6
file_only       f1_04_loop_owned_count       seed=1 latency=42.67s input=1570 output=183 api=2


## Conclusion

The first complete Action Gating ablation is now usable:

- Q: all three configs achieved 30/30 on F1.
- C: `terminal_only` is cheapest in mean input/output tokens and API calls.
- L: `baseline` remains fastest at p50; `terminal_only` is second; `file_only` has the worst p95/max.
- R: no retry/fallback/budget/stuck signals appeared in these records.

Recommended next steps:

1. Increase seeds from 3 to 5-7 to see whether the latency ordering holds.
2. Add an F2 minimal subset, because F1 is read-only code QA and may overstate the value of narrow tool exposure.
3. Add one Transition Control ablation, for example `max_iterations=10` vs `30` or terminal timeout `20s` vs `60s`.
4. Keep the repository-scope runner prompt in place for benchmark runs, because otherwise WSL environment hints pollute terminal-only results with host-filesystem search.